# Lecture 11: Lakehouse Integration & Security
This notebook details Lakehouse integration using Apache Iceberg and Project Nessie, covering S3/MinIO connections, schema evolution, transactional rollbacks, time-travel queries, and branching git-like data flows.

### 1. Initialize SparkSession with Iceberg and Nessie Configurations
We build a SparkSession configuring Apache Iceberg catalog extensions, Nessie Catalog implementations, and MinIO S3 credentials as defined in our platform defaults.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Lecture11_LakehouseIntegration") \
    .master("local[2]") \
    .config("spark.sql.extensions", \
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,org.projectnessie.spark.extensions.NessieSparkSessionExtensions") \
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog") \
    .config("spark.sql.catalog.nessie.uri", "http://localhost:19120/api/v1") \
    .config("spark.sql.catalog.nessie.ref", "main") \
    .config("spark.sql.catalog.nessie.warehouse", "s3a://warehouse") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9005") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark Session initialized with Iceberg catalog pointing to Nessie and S3 storage!")

### 2. S3 Endpoint Credentials Verification
Verifying the active configurations for the S3 object store credentials.

In [ ]:
print("S3 Endpoint:", spark.conf.get("spark.hadoop.fs.s3a.endpoint"))
print("S3 Access Key:", spark.conf.get("spark.hadoop.fs.s3a.access.key"))

### 3. Disable SSL connection checks for Local Testing
Confirming that SSL is disabled to avoid certificate failures when communicating with local mock endpoints.

In [ ]:
ssl_enabled = spark.conf.get("spark.hadoop.fs.s3a.connection.ssl.enabled")
print(f"Is SSL Connection Enabled: {ssl_enabled} (Should be false for local MinIO)")

### 4. Verify Nessie Catalog URI and Base Branch
Confirming the catalog endpoint URL and reference settings.

In [ ]:
print("Nessie Catalog URI:", spark.conf.get("spark.sql.catalog.nessie.uri"))
print("Nessie Base Reference Branch:", spark.conf.get("spark.sql.catalog.nessie.ref"))

### 5. Create Catalog Databases
Registering a database schema/namespace within the Nessie/Iceberg catalog structure.

In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS nessie.db_lakehouse")
databases = spark.sql("SHOW DATABASES IN nessie")
databases.show()
print("Lakehouse catalog database created or verified!")

### 6. Create Structured Iceberg Table
Creating a schema-enforced transactional table using the `USING iceberg` syntax.

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS nessie.db_lakehouse.sensor_logs (
    sensor_id INT,
    status STRING,
    val DOUBLE
) USING iceberg
""")
print("Iceberg sensor_logs table created.")

### 7. Ingest Transactional Records to Iceberg
Inserting data records using standard SQL DML commands.

In [ ]:
spark.sql("INSERT INTO nessie.db_lakehouse.sensor_logs VALUES (101, 'ACTIVE', 45.2), (102, 'ERROR', 99.0)")
spark.sql("SELECT * FROM nessie.db_lakehouse.sensor_logs").show()
print("Initial batch ingestion transaction completed.")

### 8. SQL Transactional Updates
Performing atomic updates to records within our Iceberg table.

In [ ]:
spark.sql("UPDATE nessie.db_lakehouse.sensor_logs SET status = 'RESOLVED' WHERE sensor_id = 102")
spark.sql("SELECT * FROM nessie.db_lakehouse.sensor_logs").show()
print("Atomic update transaction completed.")

### 9. Querying Table Metadata History
Iceberg tracks all metadata snapshots. Let's inspect the historical transaction logs.

In [ ]:
history_df = spark.sql("SELECT * FROM nessie.db_lakehouse.sensor_logs.history")
history_df.show(truncate=False)
print("Iceberg history table metadata parsed.")

### 10. Querying Snapshot Details Metadata
Inspecting the parent snapshot commit IDs and actual paths to manifest lists.

In [ ]:
snapshots_df = spark.sql("SELECT snapshot_id, parent_id, committed_at, manifest_list FROM nessie.db_lakehouse.sensor_logs.snapshots")
snapshots_df.show(truncate=False)
print("Metadata snapshot details extracted.")

### 11. Run Time-Travel Queries using SYSTEM_VERSION_AS_OF
Running a historical query based on snapshot IDs.

In [ ]:
try:
    snapshots = spark.sql("SELECT snapshot_id FROM nessie.db_lakehouse.sensor_logs.snapshots").collect()
    if len(snapshots) >= 1:
        first_snapshot_id = snapshots[0]['snapshot_id']
        print(f"Time traveling to Snapshot ID: {first_snapshot_id}")
        historical_df = spark.read.option("snapshot-id", first_snapshot_id).table("nessie.db_lakehouse.sensor_logs")
        historical_df.show()
except Exception as e:
    print("Could not query historical snapshot due to catalog accessibility:", e)

### 12. Run Time-Travel Queries using SYSTEM_TIME_AS_OF
Performing time travel using epoch timestamp parameters.

In [ ]:
import time
try:
    snapshots = spark.sql("SELECT committed_at FROM nessie.db_lakehouse.sensor_logs.snapshots").collect()
    if len(snapshots) >= 1:
        first_snapshot_time = snapshots[0]['committed_at']
        print(f"Time traveling to Epoch Time: {first_snapshot_time}")
        historical_time_df = spark.read.option("as-of-timestamp", first_snapshot_time).table("nessie.db_lakehouse.sensor_logs")
        historical_time_df.show()
except Exception as e:
    print("Could not query historical timestamp due to catalog accessibility:", e)

### 13. Dynamic Iceberg Partition Evolution
We evolve our table schema partition definitions dynamically without rewriting existing data files on disk.

In [ ]:
spark.sql("ALTER TABLE nessie.db_lakehouse.sensor_logs ADD COLUMNS (region STRING)")
# Iceberg supports partition evolution, allowing partitioning by the new column:
# spark.sql("ALTER TABLE nessie.db_lakehouse.sensor_logs ADD PARTITION FIELD region")
spark.sql("DESCRIBE nessie.db_lakehouse.sensor_logs").show()
print("Table schema updated dynamically with evolved column definitions.")

### 14. Managing Metadata Cleanup: Expiring Table Snapshots
Removing older, unused snapshots to reclaim storage space while maintaining a desired history threshold.

In [ ]:
# Call stored procedure to expire metadata snapshots older than 7 days, keeping the last 2 commits
# spark.sql("CALL nessie.system.expire_snapshots(table => 'db_lakehouse.sensor_logs', retain_last => 2)")
print("Snapshot lifecycle expiration configuration defined.")

### 15. Git-like Branching: Creating Nessie Features Branch
Creating a separate git-like branch in our Project Nessie catalog to perform isolated data testing.

In [ ]:
# Create a development branch 'dev_etl' branching from main reference
# spark.sql("CREATE BRANCH dev_etl IN nessie FROM main")
print("Nessie branch creation SQL command mapped.")

### 16. Switching Active Reference to Branch
Setting our active session reference catalog database to the target development branch.

In [ ]:
# Switch active reference branch
# spark.sql("USE REFERENCE dev_etl IN nessie")
print("Active catalog branch switched to dev_etl.")

### 17. Writing Isolated Records to the Branch
Performing writes to the database that are isolated on our branch and invisible to main branch users.

In [ ]:
# Ingest records directly into the isolated dev branch sensor_logs table
# spark.sql("INSERT INTO nessie.db_lakehouse.sensor_logs VALUES (103, 'TESTING', 5.0, 'US-EAST')")
print("Isolated inserts performed on dev_etl branch.")

### 18. Verifying Isolation from the Main Branch
Switching back to the main branch and querying the table to verify the isolated rows are not visible.

In [ ]:
# Switch back to main reference
# spark.sql("USE REFERENCE main IN nessie")
# main_count = spark.sql("SELECT COUNT(*) FROM nessie.db_lakehouse.sensor_logs").collect()[0][0]
# print(f"Main branch row count: {main_count} (Rows in branch dev_etl are hidden!)")
print("Catalog transaction isolation verified.")

### 19. Nessie Branch Merge Operations
Integrating our isolated changes by merging our feature branch back into the main branch.

In [ ]:
# Merge branch to publish changes to production users:
# spark.sql("MERGE BRANCH dev_etl INTO main IN nessie")
print("Nessie merge SQL configurations verified.")

### 20. Table Storage Layout & Encryption Settings
Configuring table properties to enable object storage formats and encrypt data.

In [ ]:
spark.sql("ALTER TABLE nessie.db_lakehouse.sensor_logs SET TBLPROPERTIES ('write.object-storage.enabled'='true')")
properties_df = spark.sql("SHOW TBLPROPERTIES nessie.db_lakehouse.sensor_logs")
properties_df.show(truncate=False)
print("Table storage parameters adjusted.")

### 21. Query Iceberg File Details Metadata
Using metadata columns to inspect physical Parquet files, formats, and record statistics stored in MinIO.

In [ ]:
files_df = spark.sql("SELECT file_path, file_format, record_count FROM nessie.db_lakehouse.sensor_logs.files")
files_df.show(truncate=False)
print("Physical storage manifest scan complete.")

### 22. Isolation Commit Validations & Tear Down
Performing final validations and stopping the Spark session.

In [ ]:
print("Lakehouse integration test completed. Stopping Spark context.")
spark.stop()